## Hyperparameter optimization

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import pmdarima as pm
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore")

# Notebook lives in ARIMA/, data/ is its sibling
DATA_ROOT   = os.path.join("..", "data", "resampled")
SPLIT_LABEL = "2025-01-01"   # only this fixed-date split

INTERVALS   = ["1min", "10min", "1h", "1d"]
FREQ_MAP    = {"1min":"T", "10min":"10T", "1h":"H", "1d":"D"}

N_JOBS      = -1  # for auto_arima
CV_FOLDS    = 3

results = []

# -----------------------------------------------------------------------------
# Run ARIMA tuning & evaluation for the single split
# -----------------------------------------------------------------------------
print(f"\n===== SPLIT: {SPLIT_LABEL} =====")
for iv in INTERVALS:
    print(f"\n--- Interval: {iv} ---")

    # load train/test directly from data/resampled/
    train_df = pd.read_parquet(os.path.join(DATA_ROOT, f"train_{iv}.parquet"))
    test_df  = pd.read_parquet(os.path.join(DATA_ROOT, f"test_{iv}.parquet"))
    train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
    test_df ["timestamp"] = pd.to_datetime(test_df["timestamp"])

    for coin in train_df["coin_id"].unique():
        print(f"\nCoin: {coin}")

        # build the training log-return series
        ts_tr = (
            train_df[train_df.coin_id == coin]
            .set_index("timestamp")["close"]
            .asfreq(FREQ_MAP[iv])
            .dropna()
        )
        lr_tr = np.log(ts_tr).diff().dropna()

        # pick order
        if iv == "1min":
            p, d, q = 1, 0, 1
            print("  Using fixed ARIMA order (1,0,1) for 1min interval")
        else:
            model = pm.auto_arima(
                lr_tr,
                start_p=0, start_q=0,
                max_p=3, max_q=3,
                seasonal=False,
                stepwise=True,
                n_jobs=N_JOBS,
                error_action="ignore",
                suppress_warnings=True,
                information_criterion="aic"
            )
            p, d, q = model.order
            print(f"  Selected ARIMA order=(p,d,q)=({p},{d},{q}), AIC={model.aic():.2f}")

        # cross‐validation on train
        tscv    = TimeSeriesSplit(n_splits=CV_FOLDS)
        cv_rmses = []
        for fold, (train_idx, val_idx) in enumerate(tscv.split(lr_tr), start=1):
            tr_fold  = lr_tr.iloc[train_idx]
            val_fold = lr_tr.iloc[val_idx]
            m = pm.ARIMA(order=(p, d, q)).fit(
                tr_fold,
                enforce_stationarity=False,
                enforce_invertibility=False,
                suppress_warnings=True
            )
            preds = m.predict(n_periods=len(val_fold))
            rmse  = np.sqrt(mean_squared_error(val_fold, preds))
            cv_rmses.append(rmse)
            print(f"    Fold {fold} RMSE={rmse:.4f}")
        cv_mean, cv_std = np.mean(cv_rmses), np.std(cv_rmses)
        print(f"  CV RMSE = {cv_mean:.4f} ± {cv_std:.4f}")

        # final fit & test evaluation
        final_model = pm.ARIMA(order=(p, d, q)).fit(
            lr_tr,
            enforce_stationarity=False,
            enforce_invertibility=False,
            suppress_warnings=True
        )
        ts_te = (
            test_df[test_df.coin_id == coin]
            .set_index("timestamp")["close"]
            .asfreq(FREQ_MAP[iv])
            .dropna()
        )
        lr_te      = np.log(ts_te).diff().dropna()
        preds_test = final_model.predict(n_periods=len(lr_te))
        rmse_test  = np.sqrt(mean_squared_error(lr_te, preds_test))
        print(f"  Test RMSE={rmse_test:.4f}")

        # record
        results.append({
            "split":     SPLIT_LABEL,
            "interval":  iv,
            "coin":      coin,
            "p":         p,
            "d":         d,
            "q":         q,
            "aic":       final_model.aic(),
            "cv_rmse":   cv_mean,
            "cv_std":    cv_std,
            "test_rmse": rmse_test
        })

# -----------------------------------------------------------------------------
# Summarize & save
# -----------------------------------------------------------------------------
df_res = pd.DataFrame(results)
print("\n=== Summary ===")
print(df_res)

# save directly in current ARIMA/ folder without nested path
df_res.to_csv("ARIMA_hyper_para.csv", index=False)


===== SPLIT: 2025-01-01 =====

--- Interval: 1min ---

Coin: BNBUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0010
    Fold 2 RMSE=0.0007
    Fold 3 RMSE=0.0009
  CV RMSE = 0.0009 ± 0.0001
  Test RMSE=0.0008

Coin: BTCUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0009
    Fold 2 RMSE=0.0006
    Fold 3 RMSE=0.0007
  CV RMSE = 0.0007 ± 0.0001
  Test RMSE=0.0008

Coin: ETHUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0012
    Fold 2 RMSE=0.0006
    Fold 3 RMSE=0.0009
  CV RMSE = 0.0009 ± 0.0002
  Test RMSE=0.0012

Coin: SOLUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0018
    Fold 2 RMSE=0.0014
    Fold 3 RMSE=0.0013
  CV RMSE = 0.0015 ± 0.0002
  Test RMSE=0.0015

Coin: XRPUSDT
  Using fixed ARIMA order (1,0,1) for 1min interval
    Fold 1 RMSE=0.0013
    Fold 2 RMSE=0.0010
    Fold 3 RMSE=0.0013
  CV RMSE = 0.0012 ± 0.0001
  Test RMSE=0.0016

--- Interval: 10min ---

Coin

## Data Prep for P&L backtest

In [2]:
import os
import warnings

import numpy as np
import pandas as pd
import pmdarima as pm
from scipy.stats import mstats

# -----------------------------------------------------------------------------
# 0) suppress warnings
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1) Configuration
# -----------------------------------------------------------------------------
# Notebook lives in ARIMA/, data/ is its sibling
DATA_ROOT    = os.path.join("..", "data", "resampled")
RESULTS_ROOT = "results"   # relative to ARIMA/

INTERVALS = ["1min", "10min", "1h", "1d"]
# pandas .asfreq freq strings
FREQ_MAP  = {"1min": "T", "10min": "10T", "1h": "H", "1d": "D"}

# -----------------------------------------------------------------------------
# 2) load best (p,d,q) per coin for the 2025-01-01 split
#    CSV was saved as ARIMA_hyper_para.csv in the ARIMA/ folder
# -----------------------------------------------------------------------------
params_df   = pd.read_csv("ARIMA_hyper_para.csv")
split_label = "2025-01-01"

# -----------------------------------------------------------------------------
# 3) generate and save per‐coin backtest inputs
# -----------------------------------------------------------------------------
for iv in INTERVALS:
    print(f"\n=== Interval: {iv} ===")
    out_dir = os.path.join(RESULTS_ROOT, iv)
    os.makedirs(out_dir, exist_ok=True)

    # load train/test sets
    train_df = pd.read_parquet(os.path.join(DATA_ROOT, f"train_{iv}.parquet"))
    test_df  = pd.read_parquet(os.path.join(DATA_ROOT, f"test_{iv}.parquet"))
    train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
    test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

    # select coins with parameters for this split/interval
    cond = (params_df["split"] == split_label) & (params_df["interval"] == iv)
    coins = params_df.loc[cond, "coin"].unique()

    for coin in coins:
        print(f"  Coin: {coin}")
        row = params_df.loc[cond & (params_df["coin"] == coin)].iloc[0]
        p, d, q = int(row.p), int(row.d), int(row.q)

        # training log-returns
        ts_tr = (
            train_df[train_df.coin_id == coin]
            .set_index("timestamp")["close"]
            .asfreq(FREQ_MAP[iv])
            .dropna()
        )
        lr_tr = np.log(ts_tr).diff().dropna()

        # fit ARIMA
        model = pm.ARIMA(order=(p, d, q)).fit(
            lr_tr,
            enforce_stationarity=False,
            enforce_invertibility=False,
            suppress_warnings=True
        )

        # test log-returns and bar returns
        ts_te   = (
            test_df[test_df.coin_id == coin]
            .set_index("timestamp")["close"]
            .asfreq(FREQ_MAP[iv])
            .dropna()
        )
        lr_te   = np.log(ts_te).diff().dropna()
        bar_ret = ts_te.pct_change().shift(-1).loc[lr_te.index]

        # forecast log-returns
        preds = model.predict(n_periods=len(lr_te))

        # assemble DataFrame for backtesting
        df_bt = pd.DataFrame({
            "pred":    preds,
            "signal":  np.sign(preds),
            "bar_ret": bar_ret
        }, index=lr_te.index)

        # filter extreme jumps
        df_bt = df_bt[df_bt["bar_ret"].abs() <= 0.20]

        # save for later backtest
        df_bt.to_parquet(os.path.join(out_dir, f"{coin}.parquet"))
        print(f"    saved {coin}.parquet")


=== Interval: 1min ===
  Coin: BNBUSDT
    saved BNBUSDT.parquet
  Coin: BTCUSDT
    saved BTCUSDT.parquet
  Coin: ETHUSDT
    saved ETHUSDT.parquet
  Coin: SOLUSDT
    saved SOLUSDT.parquet
  Coin: XRPUSDT
    saved XRPUSDT.parquet

=== Interval: 10min ===
  Coin: BNBUSDT
    saved BNBUSDT.parquet
  Coin: BTCUSDT
    saved BTCUSDT.parquet
  Coin: ETHUSDT
    saved ETHUSDT.parquet
  Coin: SOLUSDT
    saved SOLUSDT.parquet
  Coin: XRPUSDT
    saved XRPUSDT.parquet

=== Interval: 1h ===
  Coin: BNBUSDT
    saved BNBUSDT.parquet
  Coin: BTCUSDT
    saved BTCUSDT.parquet
  Coin: ETHUSDT
    saved ETHUSDT.parquet
  Coin: SOLUSDT
    saved SOLUSDT.parquet
  Coin: XRPUSDT
    saved XRPUSDT.parquet

=== Interval: 1d ===
  Coin: BNBUSDT
    saved BNBUSDT.parquet
  Coin: BTCUSDT
    saved BTCUSDT.parquet
  Coin: ETHUSDT
    saved ETHUSDT.parquet
  Coin: SOLUSDT
    saved SOLUSDT.parquet
  Coin: XRPUSDT
    saved XRPUSDT.parquet
